# Georgia 2008 Presidential Elections: Data Cleaning & Preprocessing

**Goal:** Build a clean, analysis-ready county-level table for Georgia, 2008 by merging the presidential primary and presidential general election results, and then derive summary stats (party totals).

**Output**: A single CSV where each row is a county and columns include:

- Primary per-candidate vote counts (prefixed with `pri_`)
- General per-candidate vote counts (prefixed with `gen_`)
- Party totals: `rep_primary_total`, `dem_primary_total`, `rep_general_total`, `dem_general_total`, `wri_general_total`

**Last Updated**: 2025/10/11

## 0. Library Import

In [3]:
import re 
import pandas as pd
import numpy as numpy
from pathlib import Path

/Users/amourtu1934/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 1. Inputs & Parameters

In [ ]:
# GA 2008 dataset path
PRIMARY_PATH = r"../../data/raw/2008/GA/20080205__ga__primary__president.csv"
GENERAL_PATH = r"../../data/raw/2008/GA/20081104__ga__general.csv"

# Output directory
OUTPUT_PATH  = r"../../data/processed/2008/GA/"

# Analysis parameters
DISPLAY_ROWS = 10   # Number of rows to display in dataframes

## 2. Load & Filter

We load primary and general datasets separately and immediately subset to the rows we truly need:

- Restrict `office` to 'President' to avoid mixing down-ballot contests

- Remove columns that are fully missing or irrelevant post-filter (e.g., a district column that’s empty for county-level rows)

### a. Primary Election Dataset

In [5]:
# Load primary data
primary_df = pd.read_csv(PRIMARY_PATH)
primary_df.head(DISPLAY_ROWS)

,county,office,district,party,candidate,votes
0,Appling,President,NaN,Democrat,Barack Obama,561
1,Appling,President,NaN,Democrat,Bill Richardson,2
2,Appling,President,NaN,Democrat,Christopher Dodd,3
3,Appling,President,NaN,Democrat,Dennis J. Kucinich,4
4,Appling,President,NaN,Democrat,Hillary Clinton,672
5,Appling,President,NaN,Democrat,Joe Biden,11
6,Appling,President,NaN,Democrat,John Edwards,44
7,Appling,President,NaN,Democrat,Mike Gravel,1
8,Appling,President,NaN,Republican,Alan Keyes,0
9,Appling,President,NaN,Republican,Duncan Hunter,1


In [6]:
# Different values in 'office' column
primary_df["office"].value_counts()

office
President    2703
Name: count, dtype: int64

In [7]:
# Number of missing values in each column
primary_df.isna().sum()

county          0
office          0
district     2703
party           0
candidate       0
votes           0
dtype: int64

Here, we can see that `office` has only one value "president" while `district` does not have any values at all. Thus, we can drop these two columns given that they don't give any additional information for our analysis.

In [8]:
# Drop "office" and "district" columns
primary_df = primary_df.drop(columns=["office", "district"]).reset_index(drop=True)
primary_df.shape

(2703, 4)

In [9]:
# List out all the parties in the primary election data
primary_df["party"].value_counts()

party
Republican    1431
Democrat      1272
Name: count, dtype: int64

In [37]:
# Different candidate in primary election data
primary_df["candidate"].value_counts()

candidate
Barack Obama          159
Duncan Hunter         159
Rudy Giuliani         159
Ron Paul              159
Mitt Romney           159
Mike Huckabee         159
John McCain           159
Fred Thompson         159
Alan Keyes            159
Bill Richardson       159
Mike Gravel           159
John Edwards          159
Joe Biden             159
Hillary Clinton       159
Dennis J. Kucinich    159
Christopher Dodd      159
Tom Tancredo          159
Name: count, dtype: int64

In [10]:
# Final look at the cleaned primary_df
primary_df.head(DISPLAY_ROWS)

,county,party,candidate,votes
0,Appling,Democrat,Barack Obama,561
1,Appling,Democrat,Bill Richardson,2
2,Appling,Democrat,Christopher Dodd,3
3,Appling,Democrat,Dennis J. Kucinich,4
4,Appling,Democrat,Hillary Clinton,672
5,Appling,Democrat,Joe Biden,11
6,Appling,Democrat,John Edwards,44
7,Appling,Democrat,Mike Gravel,1
8,Appling,Republican,Alan Keyes,0
9,Appling,Republican,Duncan Hunter,1


In [12]:
# Shape after preprocessing
primary_df.shape

(2703, 4)

### b. General Election Dataset

In [13]:
# Load general data
general_df = pd.read_csv(GENERAL_PATH)
general_df.head(DISPLAY_ROWS)

,county,office,district,party,candidate,votes
0,APPLING,President,NaN,Republican,John McCain,5085
1,ATKINSON,President,NaN,Republican,John McCain,1941
2,BACON,President,NaN,Republican,John McCain,3089
3,BAKER,President,NaN,Republican,John McCain,828
4,BALDWIN,President,NaN,Republican,John McCain,7823
5,BANKS,President,NaN,Republican,John McCain,5120
6,BARROW,President,NaN,Republican,John McCain,17625
7,BARTOW,President,NaN,Republican,John McCain,25976
8,BEN HILL,President,NaN,Republican,John McCain,3417
9,BERRIEN,President,NaN,Republican,John McCain,4901


In [14]:
# Different values in 'office' column
general_df["office"].value_counts()

office
President                      1908
U.S. Senate                     795
Public Service Commissioner     795
State House                     437
U.S. House                      386
State Senate                    280
Vice President                    2
Name: count, dtype: int64

In [15]:
# Only keep rows where 'office' is 'President'
general_df = general_df[general_df["office"] == "President"]
general_df.shape

(1908, 6)

In [16]:
# Now, drop the "office" column as it's no longer needed
# Also, drop the district column as it's not applicable 
general_df = general_df.drop(columns=["office", "district"]).reset_index(drop=True)
general_df.head(DISPLAY_ROWS)

,county,party,candidate,votes
0,APPLING,Republican,John McCain,5085
1,ATKINSON,Republican,John McCain,1941
2,BACON,Republican,John McCain,3089
3,BAKER,Republican,John McCain,828
4,BALDWIN,Republican,John McCain,7823
5,BANKS,Republican,John McCain,5120
6,BARROW,Republican,John McCain,17625
7,BARTOW,Republican,John McCain,25976
8,BEN HILL,Republican,John McCain,3417
9,BERRIEN,Republican,John McCain,4901


In [17]:
# List out all the parties in the general election data
general_df["party"].value_counts()

party
Write-In      1590
Republican     159
Democrat       159
Name: count, dtype: int64

In [36]:
# Different candidate in general election data
general_df["candidate"].value_counts()

candidate
John McCain            159
Barack Obama           159
Bob Barr               159
Jonathan Allen         159
Chuck Baldwin          159
Brian Russell Brown    159
David C. Byrne         159
James Harris           159
Cynthia McKinney       159
Frank Moore            159
Ralph Nader            159
Michael A. Peroutka    159
Name: count, dtype: int64

In [18]:
# Final look at the cleaned general_df
general_df.head(DISPLAY_ROWS)

,county,party,candidate,votes
0,APPLING,Republican,John McCain,5085
1,ATKINSON,Republican,John McCain,1941
2,BACON,Republican,John McCain,3089
3,BAKER,Republican,John McCain,828
4,BALDWIN,Republican,John McCain,7823
5,BANKS,Republican,John McCain,5120
6,BARROW,Republican,John McCain,17625
7,BARTOW,Republican,John McCain,25976
8,BEN HILL,Republican,John McCain,3417
9,BERRIEN,Republican,John McCain,4901


In [19]:
# Shape after preprocessing
general_df.shape

(1908, 4)

## 3. Table Pivoting

We convert tall (one row per county/party/candidate) into wide (one row per county with one column per candidate). This creates the consistent schema with previous group cleaned data.

Helper functions:

- `normalize_party(s)`: in this case, we lower everything so column names are stable with other dataframes
- `candidate_token(name)`: turns “Barack Obama” -> OBAMA, “John McCain” -> MCCAIN, etc. Create a short, readable, unique token for column names
- `pivot_wide(df, prefix, key_col="county")`: Main pivot function
        
    * groups by `county` x `party` × `candidate`, sums `votes`,
    * pivots to columns named like:
        * Primary: `pri_dem_OBAMA`, `pri_rep_MCCAIN`,...
        * General: `gen_dem_OBAMA`, `gen_rep_MCCAIN`,...

    * flattens the MultiIndex into plain column strings,
    * returns one wide row per county

In [28]:
def normalize_party(s: pd.Series) -> pd.Series:
    """
    Normalize party names: Democratic -> dem, Republican -> rep
    """
    return(s.str.strip()
           .str.capitalize()
           .map({
                "Democrat"       : "dem", 
                "Republican"     : "rep",
                "Write-in"       : "wri"
               })
           .fillna(s.str.strip().str.lower()))      # For defensive purposes only, would not expect other parties

In [29]:
SUFFIXES = {
    "JR","SR","JNR","SNR",
    "II","III","IV","V","VI","VII","VIII","IX","X","XI","XII"
}

def candidate_token(name: str) -> str:
    """
    Turn John McCain -> MCCAIN, Barack Obama -> OBAMA
    Skip suffixes, keep last name/token, capitalize, and remove punctuation
    """
    if pd.isna(name):
        return "UNKNOWN"                # Defensive purposes only, would not expect missing values
    
    # Remove suffixes
    raw = str(name).strip()

    # If a comma exists, treat as 'LAST, FIRST ...'
    if "," in raw:
        last_part = raw.split(",", 1)[0]
        last_part = re.sub(r"[^A-Za-z0-9\s]+", "", last_part).strip().upper()
        tokens = last_part.split()
        return tokens[-1] if tokens else "UNKNOWN"

    # Otherwise: remove punctuation, split, then drop trailing suffixes
    tokens = re.sub(r"[^A-Za-z0-9\s]+", "", raw).strip().upper().split()
    while tokens and tokens[-1] in SUFFIXES:
        tokens.pop()
    return tokens[-1] if tokens else "UNKNOWN"

In [30]:
def pivot_wide(df: pd.DataFrame, prefix: str, key_col: str="county") -> pd.DataFrame:
    """
    Pivot the dataframe to wide format based on party and candidate
    """
    # Normalize party names
    df['party_key'] = normalize_party(df['party'])
    
    # Create candidate tokens
    df['candidate_token'] = df['candidate'].apply(candidate_token)
    
    # Create new column names based on party and candidate token
    df['new_col'] = prefix + '_' + df['party'] + '_' + df['candidate_token']
    
    # Pivot the dataframe
    pivot_df = df.pivot_table(index=key_col, 
                              columns=["party_key", "candidate_token"], 
                              values="votes", 
                              aggfunc='sum', 
                              fill_value=0)
    
    # Flatten multi-level columns
    pivot_df.columns = [f"{prefix}_{p}_{c}" for p, c in pivot_df.columns]
    
    # Reset index to turn key_col back into a column
    pivot_df = pivot_df.reset_index()
    
    return pivot_df

In [31]:
# Primary dataframe pivot
primary_pivot = pivot_wide(primary_df, prefix="pri")
primary_pivot.head(DISPLAY_ROWS)

,county,pri_dem_BIDEN,pri_dem_CLINTON,pri_dem_DODD,pri_dem_EDWARDS,pri_dem_GRAVEL,pri_dem_KUCINICH,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_rep_GIULIANI,pri_rep_HUCKABEE,pri_rep_HUNTER,pri_rep_KEYES,pri_rep_MCCAIN,pri_rep_PAUL,pri_rep_ROMNEY,pri_rep_TANCREDO,pri_rep_THOMPSON
0,Appling,11,672,3,44,1,4,561,2,8,1060,1,0,566,17,264,0,7
1,Atkinson,5,235,0,15,0,6,241,3,5,224,0,1,218,11,153,0,4
2,Bacon,2,358,4,35,2,2,190,2,5,460,2,2,336,40,157,2,1
3,Baker,9,199,4,36,0,1,320,2,1,121,0,1,133,1,49,0,3
4,Baldwin,21,1470,9,106,4,7,3548,15,26,1217,4,3,1327,144,708,2,14
5,Banks,9,693,4,64,2,6,232,1,8,1153,3,1,686,40,516,0,8
6,Barrow,15,1668,5,112,7,11,1949,12,53,3324,1,13,1947,211,1960,5,40
7,Bartow,13,3480,12,242,11,12,2909,19,87,4831,8,18,3007,489,2886,5,43
8,Ben hill,11,554,1,62,1,12,852,7,14,423,1,3,455,25,220,1,3
9,Berrien,4,654,2,61,4,3,424,2,14,631,1,2,568,35,413,2,5


In [32]:
# Primary dataframe shape after pivot
primary_pivot.shape

(159, 18)

In [ ]:
# General dataframe pivot
general_pivot = pivot_wide(general_df, prefix="gen")
general_pivot.head(DISPLAY_ROWS)

,county,gen_dem_OBAMA,gen_rep_MCCAIN,gen_wri_ALLEN,gen_wri_BALDWIN,gen_wri_BARR,gen_wri_BROWN,gen_wri_BYRNE,gen_wri_HARRIS,gen_wri_MCKINNEY,gen_wri_MOORE,gen_wri_NADER,gen_wri_PEROUTKA
0,APPLING,1846,5085,0,0,65,0,0,0,0,0,0,0
1,ATKINSON,938,1941,0,2,23,0,0,0,0,0,0,0
2,BACON,817,3089,0,1,31,0,0,0,0,0,0,0
3,BAKER,846,828,0,0,13,0,0,0,0,0,0,0
4,BALDWIN,8587,7823,0,11,109,0,0,0,1,0,6,0
5,BANKS,1027,5120,0,3,89,0,0,0,1,0,1,0
6,BARROW,6657,17625,0,13,280,0,0,0,2,0,5,0
7,BARTOW,9662,25976,0,27,410,0,0,0,1,0,15,2
8,BEN HILL,2590,3417,0,1,28,0,0,0,0,0,0,0
9,BERRIEN,1471,4901,0,3,72,0,0,0,0,0,2,0


Also, we need to ensure that only the first letter of each county name is capitalized, not the entire word.

In [43]:
# Fix capitalization in county column
general_pivot["county"] = general_pivot["county"].str.capitalize()
general_pivot.head(DISPLAY_ROWS)

,county,gen_dem_OBAMA,gen_rep_MCCAIN,gen_wri_ALLEN,gen_wri_BALDWIN,gen_wri_BARR,gen_wri_BROWN,gen_wri_BYRNE,gen_wri_HARRIS,gen_wri_MCKINNEY,gen_wri_MOORE,gen_wri_NADER,gen_wri_PEROUTKA
0,Appling,1846,5085,0,0,65,0,0,0,0,0,0,0
1,Atkinson,938,1941,0,2,23,0,0,0,0,0,0,0
2,Bacon,817,3089,0,1,31,0,0,0,0,0,0,0
3,Baker,846,828,0,0,13,0,0,0,0,0,0,0
4,Baldwin,8587,7823,0,11,109,0,0,0,1,0,6,0
5,Banks,1027,5120,0,3,89,0,0,0,1,0,1,0
6,Barrow,6657,17625,0,13,280,0,0,0,2,0,5,0
7,Bartow,9662,25976,0,27,410,0,0,0,1,0,15,2
8,Ben hill,2590,3417,0,1,28,0,0,0,0,0,0,0
9,Berrien,1471,4901,0,3,72,0,0,0,0,0,2,0


In [34]:
# General dataframe shape after pivot
general_pivot.shape

(159, 13)

## 4. Merge Dataframes

Before merging, we verify that county names match across primary and general:

In [44]:
# Check if county names match between primary_df and general_df
primary_counties = set(primary_pivot["county"].unique())
general_counties = set(general_pivot["county"].unique())
common_counties = primary_counties.intersection(general_counties)
print(f"Number of common counties: {len(common_counties)} out of {len(primary_counties)}")

Number of common counties: 159 out of 159


Great. Since we know that all counties name are matched (given we have lowered them), we don't need to perform further data preprocessing to match the county names. Thus, we can now merge them:

In [45]:
# Merge primary and general dataframes on 'county'
merged_df = primary_pivot.merge(general_pivot, on="county", how="inner").fillna(0)    # There should be no missing values to fill with 0
merged_df.head(DISPLAY_ROWS)

,county,pri_dem_BIDEN,pri_dem_CLINTON,pri_dem_DODD,pri_dem_EDWARDS,pri_dem_GRAVEL,pri_dem_KUCINICH,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_rep_GIULIANI,...,gen_wri_ALLEN,gen_wri_BALDWIN,gen_wri_BARR,gen_wri_BROWN,gen_wri_BYRNE,gen_wri_HARRIS,gen_wri_MCKINNEY,gen_wri_MOORE,gen_wri_NADER,gen_wri_PEROUTKA
0,Appling,11,672,3,44,1,4,561,2,8,...,0,0,65,0,0,0,0,0,0,0
1,Atkinson,5,235,0,15,0,6,241,3,5,...,0,2,23,0,0,0,0,0,0,0
2,Bacon,2,358,4,35,2,2,190,2,5,...,0,1,31,0,0,0,0,0,0,0
3,Baker,9,199,4,36,0,1,320,2,1,...,0,0,13,0,0,0,0,0,0,0
4,Baldwin,21,1470,9,106,4,7,3548,15,26,...,0,11,109,0,0,0,1,0,6,0
5,Banks,9,693,4,64,2,6,232,1,8,...,0,3,89,0,0,0,1,0,1,0
6,Barrow,15,1668,5,112,7,11,1949,12,53,...,0,13,280,0,0,0,2,0,5,0
7,Bartow,13,3480,12,242,11,12,2909,19,87,...,0,27,410,0,0,0,1,0,15,2
8,Ben hill,11,554,1,62,1,12,852,7,14,...,0,1,28,0,0,0,0,0,0,0
9,Berrien,4,654,2,61,4,3,424,2,14,...,0,3,72,0,0,0,0,0,2,0


In [46]:
# Statistics check on merged dataframe 
merged_df.describe()

,pri_dem_BIDEN,pri_dem_CLINTON,pri_dem_DODD,pri_dem_EDWARDS,pri_dem_GRAVEL,pri_dem_KUCINICH,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_rep_GIULIANI,pri_rep_HUCKABEE,...,gen_wri_ALLEN,gen_wri_BALDWIN,gen_wri_BARR,gen_wri_BROWN,gen_wri_BYRNE,gen_wri_HARRIS,gen_wri_MCKINNEY,gen_wri_MOORE,gen_wri_NADER,gen_wri_PEROUTKA
count,159.000000,159.000000,159.000000,159.000000,159.000000,159.00000,159.000000,159.000000,159.000000,159.000000,...,159.000000,159.000000,159.000000,159.000000,159.000000,159.000000,159.000000,159.000000,159.000000,159.000000
mean,15.962264,2075.635220,5.685535,114.522013,5.987421,13.18239,4429.226415,11.817610,45.044025,2055.811321,...,0.050314,8.817610,180.698113,0.012579,0.025157,0.125786,1.572327,0.037736,7.283019,0.144654
std,26.582541,4656.391429,7.815222,154.938487,11.713882,28.12559,14106.056171,15.965887,94.303935,3548.705960,...,0.246462,18.226027,397.541173,0.158610,0.157097,0.966149,7.385805,0.248713,19.626942,0.744911
min,0.000000,80.000000,0.000000,5.000000,0.000000,0.00000,40.000000,0.000000,0.000000,42.000000,...,0.000000,0.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,5.000000,411.000000,2.000000,32.500000,1.000000,4.00000,476.500000,4.000000,6.000000,397.000000,...,0.000000,0.000000,29.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,9.000000,804.000000,4.000000,66.000000,3.000000,7.00000,899.000000,7.000000,17.000000,927.000000,...,0.000000,3.000000,66.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,0.000000
75%,15.000000,1745.500000,6.000000,138.500000,6.500000,12.00000,1952.500000,14.000000,37.000000,1949.000000,...,0.000000,8.500000,162.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.500000,0.000000
max,230.000000,35668.000000,69.000000,1017.000000,90.000000,248.00000,113278.000000,114.000000,698.000000,27857.000000,...,2.000000,127.000000,2966.000000,2.000000,1.000000,11.000000,68.000000,2.000000,142.000000,6.000000


Now, we will add party totals columns: 

- Primary totals:
    * `rep_primary_total` = sum of all `pri_rep_*` columns
    * `dem_primary_total` = sum of all `pri_dem_*` columns

- General totals:
    * `rep_general_total` = sum of all `gen_rep_*` columns
    * `dem_general_total` = sum of all `gen_dem_*` columns
    * `wri_general_total` = sum of all `gen_wri_*` columns

In [47]:
# Add party totals for primary election
rep_primary_cols   = [c for c in merged_df.columns if c.startswith("pri_rep_")]
dem_primary_cols   = [c for c in merged_df.columns if c.startswith("pri_dem_")]

merged_df["rep_primary_total"] = merged_df[rep_primary_cols].sum(axis=1) if rep_primary_cols else 0
merged_df["dem_primary_total"] = merged_df[dem_primary_cols].sum(axis=1) if dem_primary_cols else 0

In [48]:
# Add party totals for general election
rep_general_cols   = [c for c in merged_df.columns if c.startswith("gen_rep_")]
dem_general_cols   = [c for c in merged_df.columns if c.startswith("gen_dem_")]
wri_general_cols   = [c for c in merged_df.columns if c.startswith("gen_wri_")]

merged_df["rep_general_total"] = merged_df[rep_general_cols].sum(axis=1) if rep_general_cols else 0
merged_df["dem_general_total"] = merged_df[dem_general_cols].sum(axis=1) if dem_general_cols else 0
merged_df["wri_general_total"] = merged_df[wri_general_cols].sum(axis=1) if wri_general_cols else 0

In [49]:
# Print out all the column names in the final dataframe
print("Final columns in the cleaned dataframe:")
merged_df.columns

Final columns in the cleaned dataframe:


Index(['county', 'pri_dem_BIDEN', 'pri_dem_CLINTON', 'pri_dem_DODD',
       'pri_dem_EDWARDS', 'pri_dem_GRAVEL', 'pri_dem_KUCINICH',
       'pri_dem_OBAMA', 'pri_dem_RICHARDSON', 'pri_rep_GIULIANI',
       'pri_rep_HUCKABEE', 'pri_rep_HUNTER', 'pri_rep_KEYES', 'pri_rep_MCCAIN',
       'pri_rep_PAUL', 'pri_rep_ROMNEY', 'pri_rep_TANCREDO',
       'pri_rep_THOMPSON', 'gen_dem_OBAMA', 'gen_rep_MCCAIN', 'gen_wri_ALLEN',
       'gen_wri_BALDWIN', 'gen_wri_BARR', 'gen_wri_BROWN', 'gen_wri_BYRNE',
       'gen_wri_HARRIS', 'gen_wri_MCKINNEY', 'gen_wri_MOORE', 'gen_wri_NADER',
       'gen_wri_PEROUTKA', 'rep_primary_total', 'dem_primary_total',
       'rep_general_total', 'dem_general_total', 'wri_general_total'],
      dtype='object')

In [50]:
# Preview merged dataframe with totals
merged_df.head(DISPLAY_ROWS)

,county,pri_dem_BIDEN,pri_dem_CLINTON,pri_dem_DODD,pri_dem_EDWARDS,pri_dem_GRAVEL,pri_dem_KUCINICH,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_rep_GIULIANI,...,gen_wri_HARRIS,gen_wri_MCKINNEY,gen_wri_MOORE,gen_wri_NADER,gen_wri_PEROUTKA,rep_primary_total,dem_primary_total,rep_general_total,dem_general_total,wri_general_total
0,Appling,11,672,3,44,1,4,561,2,8,...,0,0,0,0,0,1923,1298,5085,1846,65
1,Atkinson,5,235,0,15,0,6,241,3,5,...,0,0,0,0,0,616,505,1941,938,25
2,Bacon,2,358,4,35,2,2,190,2,5,...,0,0,0,0,0,1005,595,3089,817,32
3,Baker,9,199,4,36,0,1,320,2,1,...,0,0,0,0,0,309,571,828,846,13
4,Baldwin,21,1470,9,106,4,7,3548,15,26,...,0,1,0,6,0,3445,5180,7823,8587,127
5,Banks,9,693,4,64,2,6,232,1,8,...,0,1,0,1,0,2415,1011,5120,1027,94
6,Barrow,15,1668,5,112,7,11,1949,12,53,...,0,2,0,5,0,7554,3779,17625,6657,300
7,Bartow,13,3480,12,242,11,12,2909,19,87,...,0,1,0,15,2,11374,6698,25976,9662,455
8,Ben hill,11,554,1,62,1,12,852,7,14,...,0,0,0,0,0,1145,1500,3417,2590,29
9,Berrien,4,654,2,61,4,3,424,2,14,...,0,0,0,2,0,1671,1154,4901,1471,77


Now, we save the cleaned dataframe into the processed directory.

In [51]:
# Save the cleaned and merged dataframe to CSV
out_dir = Path(OUTPUT_PATH)
out_dir.mkdir(parents=True, exist_ok=True)
merged_df.to_csv(OUTPUT_PATH + "GA.csv", index=False)